# HomeTown AR: product photo to 3D model (TripoSR on a free Colab GPU)
1. **Runtime > Change runtime type > T4 GPU**, then run the cells in order.
2. On your computer run `node scripts/colab-batch.mjs export --next 20` and upload `models/colab/products.json` when asked.
3. Download `models.zip`, then `node scripts/colab-batch.mjs import models.zip` and `node scripts/image-to-3d.mjs --normalise`.

Each model is a guess from ONE photo (back and sides are invented). Review them in the admin dashboard before customers see them.

In [ ]:
!nvidia-smi -L
!git clone --depth 1 https://github.com/VAST-AI-Research/TripoSR /content/TripoSR
!pip -q install omegaconf==2.3.0 einops==0.7.0 transformers==4.35.0 trimesh==4.0.5 rembg onnxruntime xatlas==0.0.9 moderngl==5.10.0 imageio[ffmpeg]
!pip -q install git+https://github.com/tatsy/torchmcubes.git

In [ ]:
from google.colab import files
up = files.upload()               # choose products.json
import json, io, os, sys, zipfile, requests, numpy as np, torch, rembg
from PIL import Image
sys.path.insert(0, '/content/TripoSR')
from tsr.system import TSR
from tsr.utils import remove_background, resize_foreground
products = json.load(open('products.json'))
print(len(products), 'products')

In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
assert device != 'cpu', 'Turn on a GPU: Runtime > Change runtime type > T4 GPU'
model = TSR.from_pretrained('stabilityai/TripoSR', config_name='config.yaml', weight_name='model.ckpt')
model.renderer.set_chunk_size(8192); model.to(device)
session = rembg.new_session()
os.makedirs('out', exist_ok=True)
for p in products:
    try:
        img = Image.open(io.BytesIO(requests.get(p['photo'], timeout=60).content)).convert('RGB')
        img = resize_foreground(remove_background(img, session), 0.85)
        a = np.array(img).astype(np.float32) / 255.0
        a = a[:, :, :3] * a[:, :, 3:4] + (1 - a[:, :, 3:4]) * 0.5
        img = Image.fromarray((a * 255).astype(np.uint8))
        with torch.no_grad(): codes = model([img], device=device)
        mesh = model.extract_mesh(codes, True, resolution=256)[0]
        mesh.export(f"out/{p['id']}.glb")
        print('made', p['id'])
    except Exception as e:
        print('FAILED', p['id'], e)

In [ ]:
with zipfile.ZipFile('models.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir('out')): z.write(f'out/{f}', f)
files.download('models.zip')